# Stockfish 19 搜尋評估標註器 (`*.sf19.npz`) — Google Colab 專用版

本筆記本用於在 Google Colab 上執行 `tuner.data_pipeline.annotate_sf_labels`，將資料集局面交由 **Stockfish 19 (AVX2)** 進行固定節點搜尋，產出輕量級 sidecar 標註檔（`dataset_epd_wdl_20m.sf19.npz` 或 `val_zurichess.sf19.npz`），並**自動定期同步回 Google Drive**：
`/content/drive/MyDrive/chess_numba_data/hce/`

---

### 核心用途與標註模式 (`ANNOTATION_TARGET`)
1. **`"both"`（推薦，善用閒置算力）**：
   - 先完整標註 **`val_zurichess.npz`**（`1,079,507` 局面 -> `val_zurichess.sf19.npz`），再接續已完成的 `17.63M` 續傳 **`dataset_epd_wdl_20m.npz` 剩餘 `2M unseen holdout`**（`skip_holdout=False` -> 滿 `19.63M`）。
2. **`"zurichess"` / `"holdout_20m"` / `"20m_train_dev"`**：可依需求單獨執行其中一個目標。
3. **驗證集純 WDL 分層診斷（絕不替代勝負目標）**：
   - 替 `val_zurichess.npz` 與 `2M unseen holdout` 補齊 `sf_cp` / `sf_d1_cp` 後，`TexelTuningSession` 與 `tuner.validate_tuned` 在任何模式（**包含 `USE_SF_LABELS = False` 純 WDL 調參**）下，仍**嚴格保留 100% 實戰 WDL 勝負**作為 Gate 指標，同時額外拆出 `sf19-quiet (WDL)`（`|sf_cp - sf_d1_cp| <= 180`）與 `sf19-tactical (WDL)` 子診斷，分離戰術雜訊與真正的靜態位置評估改進。
4. **隨時中斷、無縫續傳 (`resume=True`)**：
   - 每標註 `DRIVE_SYNC_EVERY = 200,000` 筆、中斷或完成時，皆會自動將 `.sf19.npz` 與 checkpoint 備份至 Google Drive。

In [1]:
# =====================================================================
# 檢視 Colab 雲端硬體規格與 CPU 核心數
# =====================================================================
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!free -h
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.4Gi        47Gi       2.2Mi       2.7Gi        49Gi
Swap:             0B          0B          0B
Python 3.13.18


In [6]:
# 00 — [Colab 專用] 檢查硬體規格、掛載 Google Drive、同步 Repo 與載入資料集
import os
import shutil
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("=== Colab CPU & RAM 規格 ===")
    subprocess.run(["bash", "-c", "lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\\(s\\)'"], check=False)
    subprocess.run(["free", "-h"], check=False)

    drive.mount("/content/drive")
    DRIVE_HCE_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
    DRIVE_BIN_DIR = Path("/content/drive/MyDrive/chess_numba_data/bin")
    DRIVE_HCE_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_BIN_DIR.mkdir(parents=True, exist_ok=True)

    REPO_URL = "https://github.com/b08203031/chess_numba.git"
    BRANCH = "main"
    REPO_DIR = Path("/content/chess_numba")
    if not (REPO_DIR / ".git").is_dir():
        print(f"[Git] Cloning {REPO_URL} (branch={BRANCH}) -> {REPO_DIR} ...")
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        print(f"[Git] Syncing {REPO_DIR} -> origin/{BRANCH} ...")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-f", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)

    LOCAL_HCE_DIR = REPO_DIR / "tuner" / "data" / "hce"
    LOCAL_HCE_DIR.mkdir(parents=True, exist_ok=True)

    sync_files = (
        "dataset_epd_wdl_20m.npz",
        "dataset_epd_wdl_20m.sf19.npz",
        "dataset_epd_wdl_20m.sf19.ckpt_cp.npy",
        "dataset_epd_wdl_20m.sf19.ckpt_d1.npy",
        "val_zurichess.npz",
        "val_zurichess.sf19.npz",
        "val_zurichess.sf19.ckpt_cp.npy",
        "val_zurichess.sf19.ckpt_d1.npy",
    )
    for fname in sync_files:
        src = DRIVE_HCE_DIR / fname
        dst = LOCAL_HCE_DIR / fname
        if src.is_file():
            if not dst.is_file() or src.stat().st_size != dst.stat().st_size or ".sf19" in fname:
                print(f"📥 從 Google Drive 複製 {fname} ({src.stat().st_size / (1024**2):.1f} MB) -> 本機 SSD...")
                shutil.copy2(src, dst)
            else:
                print(f"✅ 本機 SSD 已存在 {fname} ({dst.stat().st_size / (1024**2):.1f} MB)")
        elif fname in ("dataset_epd_wdl_20m.npz", "val_zurichess.npz"):
            print(f"⚠️ Google Drive 尚無 {fname}（若下方 ANNOTATION_TARGET 會用到它，請先上傳至 chess_numba_data/hce/）。")
        elif fname.endswith(".sf19.npz"):
            print(f"ℹ️ Google Drive 尚無既有 {fname}，將建立全新標註檔。")
else:
    print("ℹ️ 目前不在 Google Colab 環境，將使用本機路徑執行。")

=== Colab CPU & RAM 規格 ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Git] Syncing /content/chess_numba -> origin/feat-initial-bitboard-setup ...
📥 從 Google Drive 複製 dataset_epd_wdl_20m.npz (674.2 MB) -> 本機 SSD...
ℹ️ Google Drive 尚無既有 dataset_epd_wdl_20m.sf19.npz，將建立全新標註檔。


In [7]:
# 1 — 自動準備 Stockfish 19 (Linux x86-64 AVX2) 並快取至 Google Drive
import os
import platform
import shutil
import stat
import subprocess
import tarfile
import urllib.request
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "chess_engine").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

EXT_SF_DIR = REPO_ROOT / "external" / "stockfish"
EXT_SF_DIR.mkdir(parents=True, exist_ok=True)

if platform.system() == "Windows":
    SF_BIN_PATH = EXT_SF_DIR / "stockfish-windows-x86-64-avx2.exe"
else:
    SF_BIN_PATH = EXT_SF_DIR / "stockfish-ubuntu-x86-64-avx2"
    DRIVE_CACHED_SF = Path("/content/drive/MyDrive/chess_numba_data/bin/stockfish-ubuntu-x86-64-avx2")

    if not SF_BIN_PATH.is_file() and DRIVE_CACHED_SF.is_file():
        print(f"⚡ 從 Google Drive 快取載入 Stockfish 執行檔: {DRIVE_CACHED_SF}")
        shutil.copy2(DRIVE_CACHED_SF, SF_BIN_PATH)
        SF_BIN_PATH.chmod(SF_BIN_PATH.stat().st_mode | stat.S_IEXEC)

    if not SF_BIN_PATH.is_file():
        SF_URL = "https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar"
        tar_path = EXT_SF_DIR / "sf_avx2.tar"
        try:
            print(f"🌐 下載官方 Stockfish Linux AVX2 發行版: {SF_URL} ...")
            urllib.request.urlretrieve(SF_URL, tar_path)
            with tarfile.open(tar_path, "r") as tf:
                for member in tf.getmembers():
                    if member.isfile() and "stockfish" in Path(member.name).name:
                        extracted = tf.extractfile(member)
                        if extracted is not None:
                            with open(SF_BIN_PATH, "wb") as out_f:
                                shutil.copyfileobj(extracted, out_f)
                            break
            tar_path.unlink(missing_ok=True)
            SF_BIN_PATH.chmod(SF_BIN_PATH.stat().st_mode | stat.S_IEXEC)
        except Exception as dl_err:
            print(f"⚠️ 官方發行版下載失敗 ({dl_err})，改從官方原始碼編譯 AVX2 版本...")
            sf_src_dir = Path("/tmp/Stockfish_build")
            if not sf_src_dir.exists():
                subprocess.run(["git", "clone", "--depth", "1", "https://github.com/official-stockfish/Stockfish.git", str(sf_src_dir)], check=True)
            nproc = str(os.cpu_count() or 4)
            subprocess.run(["make", f"-j{nproc}", "build", "ARCH=x86-64-avx2"], cwd=str(sf_src_dir / "src"), check=True)
            shutil.copy2(sf_src_dir / "src" / "stockfish", SF_BIN_PATH)
            SF_BIN_PATH.chmod(SF_BIN_PATH.stat().st_mode | stat.S_IEXEC)

        if DRIVE_CACHED_SF.parent.exists() and SF_BIN_PATH.is_file():
            shutil.copy2(SF_BIN_PATH, DRIVE_CACHED_SF)
            print(f"💾 已將 Stockfish 執行檔快取至 Google Drive: {DRIVE_CACHED_SF}")

# 驗證 UCI 啟動與版本資訊
res = subprocess.run([str(SF_BIN_PATH)], input="uci\nquit\n", text=True, capture_output=True, timeout=10, check=True)
banner = res.stdout.splitlines()[0] if res.stdout else "Unknown"
print(f"✅ Stockfish 準備完成: {banner} ({SF_BIN_PATH})")

🌐 下載官方 Stockfish Linux AVX2 發行版: https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar ...
⚠️ 官方發行版下載失敗 (HTTP Error 404: Not Found)，改從官方原始碼編譯 AVX2 版本...
💾 已將 Stockfish 執行檔快取至 Google Drive: /content/drive/MyDrive/chess_numba_data/bin/stockfish-ubuntu-x86-64-avx2
✅ Stockfish 準備完成: Stockfish dev-20261110-def601cc by the Stockfish developers (see AUTHORS file) (/content/chess_numba/external/stockfish/stockfish-ubuntu-x86-64-avx2)


In [8]:
# 2 — 執行 Stockfish 19 搜尋標註 (支援隨時中斷 + 自動定期同步 Google Drive)
import inspect
import os
import shutil
import threading
from pathlib import Path
from tuner.data_pipeline.annotate_sf_labels import annotate_dataset

# ==================== 標註目標與參數設定 ====================
# ANNOTATION_TARGET:
#   - "both"         : 先標註 val_zurichess.npz (1.08M)，再續傳完成 dataset_epd_wdl_20m.npz 剩餘 2M holdout
#   - "zurichess"    : 僅標註 val_zurichess.npz -> val_zurichess.sf19.npz
#   - "holdout_20m"  : 接續既有的 dataset_epd_wdl_20m.sf19.npz，補齊最後 2M holdout (SKIP_HOLDOUT=False)
#   - "20m_train_dev": 僅標註 dataset_epd_wdl_20m.npz 的 dev + train (SKIP_HOLDOUT=True)
ANNOTATION_TARGET = "both"

# NODES: 每個局面的固定搜尋節點數（沿用 5000 與先前 17.63M 標註深度完全對齊）
NODES = 5000

# LIMIT: 0 = 全部目標；若只想先小批量測試設正整數
LIMIT = 0

WORKERS = os.cpu_count() or 8  # Colab 上直接使用全部 vCPU 核心數
BATCH_SIZE = 1024
SAVE_EVERY = 100_000          # 本機 SSD memmap 刷新頻率
DRIVE_SYNC_EVERY = 200_000   # 每標註 20 萬筆自動備份 .sf19.npz 到 Google Drive
# ======================================================

HCE_DIR = Path("tuner/data/hce")
DRIVE_HCE_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
HAS_DRIVE = Path("/content/drive/MyDrive").exists()

_TARGETS = {
    "zurichess": [
        (HCE_DIR / "val_zurichess.npz", HCE_DIR / "val_zurichess.sf19.npz", False),
    ],
    "holdout_20m": [
        (HCE_DIR / "dataset_epd_wdl_20m.npz", HCE_DIR / "dataset_epd_wdl_20m.sf19.npz", False),
    ],
    "20m_train_dev": [
        (HCE_DIR / "dataset_epd_wdl_20m.npz", HCE_DIR / "dataset_epd_wdl_20m.sf19.npz", True),
    ],
    "both": [
        (HCE_DIR / "val_zurichess.npz", HCE_DIR / "val_zurichess.sf19.npz", False),
        (HCE_DIR / "dataset_epd_wdl_20m.npz", HCE_DIR / "dataset_epd_wdl_20m.sf19.npz", False),
    ],
}
if ANNOTATION_TARGET not in _TARGETS:
    raise ValueError(f"Unknown ANNOTATION_TARGET={ANNOTATION_TARGET!r}, expected one of {list(_TARGETS)}")

jobs = _TARGETS[ANNOTATION_TARGET]
missing = [str(ds) for ds, _, _ in jobs if not ds.is_file()]
if missing:
    raise FileNotFoundError(
        "找不到下列目標資料集，請確認已放入 tuner/data/hce/ 或 Google Drive chess_numba_data/hce/：\n  "
        + "\n  ".join(missing)
    )

def _sync_sidecar_to_drive(local_out: Path) -> None:
    if not HAS_DRIVE:
        return
    DRIVE_HCE_DIR.mkdir(parents=True, exist_ok=True)
    for f in (
        local_out,
        local_out.with_suffix(".ckpt_cp.npy"),
        local_out.with_suffix(".ckpt_d1.npy"),
    ):
        if f.is_file():
            try:
                tmp_dst = DRIVE_HCE_DIR / f"{f.name}.tmp"
                shutil.copy2(f, tmp_dst)
                tmp_dst.replace(DRIVE_HCE_DIR / f.name)
            except Exception:
                pass

sig_params = inspect.signature(annotate_dataset).parameters
for ds_path, local_out, skip_hold in jobs:
    print(f"\n🚀 === 開始標註 {ds_path.name} -> {local_out.name} (nodes={NODES}, skip_holdout={skip_hold}) ===")
    drive_out = (DRIVE_HCE_DIR / local_out.name) if HAS_DRIVE else None
    stop_sync = threading.Event()

    def _periodic_drive_watcher(target_out: Path = local_out, ev: threading.Event = stop_sync) -> None:
        while not ev.wait(180.0):
            _sync_sidecar_to_drive(target_out)

    watcher = threading.Thread(target=_periodic_drive_watcher, daemon=True)
    watcher.start()

    extra_kwargs = {}
    if "drive_output_path" in sig_params:
        extra_kwargs["drive_output_path"] = drive_out
        extra_kwargs["drive_sync_every"] = DRIVE_SYNC_EVERY

    try:
        annotate_dataset(
            dataset_path=ds_path,
            output_path=local_out,
            stockfish_path=str(SF_BIN_PATH),
            nodes=NODES,
            workers=WORKERS,
            batch_size=BATCH_SIZE,
            save_every=SAVE_EVERY,
            limit=LIMIT,
            skip_holdout=skip_hold,
            resume=True,
            hash_mb=64,
            **extra_kwargs,
        )
    finally:
        stop_sync.set()
        _sync_sidecar_to_drive(local_out)
        if drive_out is not None and drive_out.is_file():
            print(f"☁️ 已同步最新標註檔與 Checkpoint 至 Google Drive: {drive_out}")

=== Stockfish Mixed-Label Sidecar Annotator ===
  Dataset   : tuner/data/hce/dataset_epd_wdl_20m.npz
  Output    : tuner/data/hce/dataset_epd_wdl_20m.sf19.npz
  Drive Sync: /content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz (every 200,000 rows)
  Engine    : Stockfish dev-20261110-def601cc by the Stockfish developers (see AUTHORS file) (/content/chess_numba/external/stockfish/stockfish-ubuntu-x86-64-avx2)
  Search    : 'go nodes 5000' | workers=8 | hash=64 MB/worker
  Loaded 19,627,749 bitboard rows in 7.5s
  Target pool: 17,627,749 rows (already done: 0, remaining todo: 17,627,749)
  [100.00%] +17,627,749/17,627,749 | 687 pos/s | ETA   0.0 min
Saved sidecar: tuner/data/hce/dataset_epd_wdl_20m.sf19.npz (55.93 MB) | annotated=17,627,749/19,627,749 (89.81%)
Synced to Drive: /content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz (55.93 MB)
  sf_cp stats : mean=+9.2 cp, std=2721.3 cp, p10=-480, p90=+492
  |sf - d1|   : median=38 cp, p90=200 cp, >180c

In [10]:
# 3 — 檢視 .sf19.npz 標註覆蓋率、sf19-quiet 純 WDL 切分比例與 K_sf 預覽
import numpy as np
from pathlib import Path
from tuner.tuner import split_dataset_indices, fit_best_k
from tuner.data_pipeline.annotate_sf_labels import (
    UNANNOTATED_CP,
    compute_quiet_and_tactical_masks,
)

HCE_DIR = Path("tuner/data/hce")
DRIVE_HCE_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
MAX_TACTICAL_GAP = 180

def _inspect_sidecar(ds_path: Path, sidecar_path: Path, is_main_20m: bool) -> None:
    if not ds_path.is_file() or not sidecar_path.is_file():
        print(f"ℹ️ 略過 {sidecar_path.name}（找不到本機資料或 sidecar 檔）")
        return
    with np.load(ds_path) as ds:
        wdl_results = ds["results"].astype(np.float64)
    with np.load(sidecar_path) as sf:
        sf_cp = sf["sf_cp"]
        sf_d1_cp = sf["sf_d1_cp"] if "sf_d1_cp" in sf.files else None
        nodes_meta = int(sf["nodes"]) if "nodes" in sf.files else -1
        banner_meta = str(sf["sf_banner"]) if "sf_banner" in sf.files else "Stockfish"

    n_total = len(wdl_results)
    ann, quiet, tactical = compute_quiet_and_tactical_masks(
        sf_cp, sf_d1_cp, max_tactical_gap=MAX_TACTICAL_GAP
    )
    drive_sc = DRIVE_HCE_DIR / sidecar_path.name

    print(f"=== Sidecar 診斷報告 ({sidecar_path.name}) ===")
    print(f"  標註引擎        : {banner_meta} (nodes={nodes_meta})")
    print(f"  本機檔案大小    : {sidecar_path.stat().st_size / (1024**2):.2f} MB")
    if drive_sc.is_file():
        print(f"  雲端硬碟備份    : ✅ {drive_sc} ({drive_sc.stat().st_size / (1024**2):.2f} MB)")
    n_ann = int(np.count_nonzero(ann))
    n_q = int(np.count_nonzero(quiet))
    n_t = int(np.count_nonzero(tactical))
    print(f"  總覆蓋率        : {n_ann:,} / {n_total:,} ({100.0 * n_ann / max(n_total, 1):.2f}%)")
    if n_ann > 0:
        print(
            f"  Quiet / Tactical: quiet={n_q:,} ({100.0 * n_q / n_ann:.2f}%) | "
            f"tactical(>|{MAX_TACTICAL_GAP}|cp)={n_t:,} ({100.0 * n_t / n_ann:.2f}%)"
        )

    if is_main_20m:
        train_idx, dev_idx, holdout_idx = split_dataset_indices(n_total, 1_000_000, 2_000_000)
        dev_valid = ann[dev_idx]
        train_valid = ann[train_idx]
        train_10m_valid = ann[train_idx[:10_000_000]]
        print(f"  Dev 覆蓋率      : {int(np.sum(dev_valid)):,} / {len(dev_idx):,} ({100.0 * np.mean(dev_valid):.2f}%)")
        print(f"  Train(前10M)覆蓋: {int(np.sum(train_10m_valid)):,} / 10,000,000 ({100.0 * np.mean(train_10m_valid):.2f}%)")
        print(f"  Train(全部)覆蓋 : {int(np.sum(train_valid)):,} / {len(train_idx):,} ({100.0 * np.mean(train_valid):.2f}%)")
        if holdout_idx is not None:
            hold_valid = ann[holdout_idx]
            hold_q = int(np.count_nonzero(quiet[holdout_idx]))
            hold_t = int(np.count_nonzero(tactical[holdout_idx]))
            print(
                f"  Holdout 覆蓋率  : {int(np.sum(hold_valid)):,} / {len(holdout_idx):,} "
                f"({100.0 * np.mean(hold_valid):.2f}%) [quiet={hold_q:,}, tactical={hold_t:,}]"
            )
        sample_valid = dev_idx[dev_valid]
    else:
        sample_valid = np.flatnonzero(ann)

    if len(sample_valid) > 0:
        idx_s = sample_valid[:200_000]
        k_sf, mse_sf = fit_best_k(sf_cp[idx_s].astype(np.float64), wdl_results[idx_s])
        print(f"  預估自動校準 K_sf: {k_sf:.4f} (SF19 對實戰 WDL MSE = {mse_sf:.6f})\n")

_inspect_sidecar(HCE_DIR / "val_zurichess.npz", HCE_DIR / "val_zurichess.sf19.npz", is_main_20m=False)
_inspect_sidecar(HCE_DIR / "dataset_epd_wdl_20m.npz", HCE_DIR / "dataset_epd_wdl_20m.sf19.npz", is_main_20m=True)

print("🚀 下一步：")
print("  1. 直接打開 texel_tuner_runner.ipynb 執行 # 00（會自動從 Google Drive 複製 .sf19.npz）與 # 0。")
print("  2. 在 USE_SF_LABELS = False 純 WDL 調參時，val_zurichess 與 2M holdout 的 sf19-quiet (WDL) 診斷也會自動啟用！")

DOWNLOAD_TO_LOCAL = False
if DOWNLOAD_TO_LOCAL:
    try:
        from google.colab import files  # type: ignore
        for sc in (HCE_DIR / "val_zurichess.sf19.npz", HCE_DIR / "dataset_epd_wdl_20m.sf19.npz"):
            if sc.is_file():
                files.download(str(sc))
    except ImportError:
        print("目前不在 Colab 環境，檔案已位於:", HCE_DIR.resolve())

=== Sidecar 診斷報告 (dataset_epd_wdl_20m.sf19.npz) ===
  標註引擎        : Stockfish dev-20261110-def601cc by the Stockfish developers (see AUTHORS file) (nodes=5000)
  本機檔案大小    : 55.93 MB
  雲端硬碟備份    : ✅ 已同步至 /content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz (55.93 MB)
  Dev 覆蓋率      : 1,000,000 / 1,000,000 (100.00%)
  Train(前10M)覆蓋: 10,000,000 / 10,000,000 (100.00%)
  Train(全部)覆蓋 : 16,627,749 / 16,627,749 (100.00%)
  預估自動校準 K_sf: 0.6546 (SF19 對實戰 WDL MSE = 0.099263)

🚀 下一步：
  1. 若要在 Colab 上調參：直接打開 texel_tuner_runner.ipynb，執行 # 00 會自動從 Google Drive 載入此 .sf19.npz，並在 # 2 設 USE_SF_LABELS = True！
  2. 若要下載回本機電腦調參：將下方 DOWNLOAD_TO_LOCAL 改為 True 執行，或直接從 Google Drive (chess_numba_data/hce/) 下載 dataset_epd_wdl_20m.sf19.npz 放到本機 tuner/data/hce/。


In [11]:
from google.colab import runtime

# 執行這行會直接中斷並釋放目前使用的虛擬機器運算資源
runtime.unassign()
